## 1. Setup & GPU Check

We verify that PyTorch can access the GPU. All training will use CUDA if available.


In [1]:
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'

print("Using device:", device)
if device == 'cuda':
    print(torch.cuda.get_device_name(0))

Using device: cuda
NVIDIA GeForce RTX 2080 SUPER


## 2. Load Dataset

We locate and load all text files from the dataset directory.


In [2]:
import glob

files = glob.glob("../data/archive/*.txt")

print("Files found:", len(files))
for f in files:
    print(f)

Files found: 5
../data/archive\The Book of Meditations.txt
../data/archive\The Book of Mormon.txt
../data/archive\The Gospel of Buddha.txt
../data/archive\The King James Bible.txt
../data/archive\The Koran.txt


## 3. Read Raw Text

We combine all files into one continuous text string.


In [3]:
text = ""

for file in files:
    with open(file, "r", encoding="utf-8") as f:
        text += f.read()

print("Total characters:", len(text))

Total characters: 8032083


## 4. Clean Text (Final Version)

We remove Gutenberg metadata, front matter (contents, notes), and normalize formatting to produce clean, continuous text suitable for training.


In [7]:
import re

def clean_gutenberg(text):
    # Remove Gutenberg header/footer
    start = text.find("*** START")
    end = text.find("*** END")
    if start != -1 and end != -1:
        text = text[start:end]
    return text


def remove_front_matter(text):
    lines = text.split("\n")
    cleaned_lines = []
    
    for line in lines:
        line_lower = line.lower().strip()
        
        if len(line_lower) < 3:
            continue
        
        if any(x in line_lower for x in [
            "project gutenberg",
            "produced by",
            "transcriber",
            "ebook",
            "http",
            "www.",
            "email",
            "copyright",
            "table of contents",
            "contents",
            "illustration",
            "preface",
            "introduction",
            "glossary",
            "appendix",
            "notes",
            "scanned by",
            "plain text version",
            "symbol.ttf",
            "ocr",
            "fonts folder"
        ]):
            continue
        
        if line.isupper() and len(line.split()) < 10:
            continue
        
        if len(line.split()) < 4:
            continue
        
        cleaned_lines.append(line)
    
    return " ".join(cleaned_lines)


def normalize_text(text):
    # Replace newlines/tabs
    text = text.replace("\n", " ")
    text = text.replace("\t", " ")
    
    # Remove weird characters but keep punctuation
    text = re.sub(r"[^a-zA-Z0-9\s.,!?;:'\"()-]", "", text)
    
    # Remove multiple spaces
    text = re.sub(r"\s+", " ", text)
    
    return text.strip()


# 🔹 Apply full cleaning pipeline
cleaned_text = ""

for file in files:
    with open(file, "r", encoding="utf-8") as f:
        raw = f.read()
        
        text = clean_gutenberg(raw)
        text = remove_front_matter(text)
        text = normalize_text(text)
        
        cleaned_text += text + " \n\n---NEW_BOOK---\n\n "

print("Cleaned characters:", len(cleaned_text))

Cleaned characters: 7568245


## 5. Inspect Cleaned Text

We verify that the dataset now contains mostly natural language with minimal noise.


In [9]:
print(cleaned_text[:500])

portions of the text have been added by hand and they will require the folder. This is a standard Windows font, so should be present on most version with the various symbols mentioned above. MARCUS AURELIUS ANTONINUS was born on April 26, A.D. 121. His real name was M. Annius Verus, and he was sprung of a noble family which claimed descent from Numa, second King of Rome. Thus the most religious of emperors came of the blood of the most pious of early kings. His father, Annius Verus, had held hig


## 6. Build Vocabulary

We convert characters into integers so the model can process text.

This is the first step in turning language into numbers.


In [11]:
chars = sorted(list(set(cleaned_text)))
vocab_size = len(chars)

print("Vocab size:", vocab_size)
print(chars[:76])

Vocab size: 76
['\n', ' ', '!', '"', "'", '(', ')', ',', '-', '.', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', ':', ';', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', '_', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']


## 7. Encode Text

We map each character to an integer (encoding) and create the full dataset tensor.


In [12]:
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}

def encode(s):
    return [stoi[c] for c in s]

def decode(l):
    return ''.join([itos[i] for i in l])

import torch

data = torch.tensor(encode(cleaned_text), dtype=torch.long)

print(data.shape)

torch.Size([7568245])


## 8. Train / Validation Split

We split the dataset so we can monitor performance during training.


In [13]:
n = int(0.9 * len(data))

train_data = data[:n]
val_data = data[n:]

print("Train size:", len(train_data))
print("Val size:", len(val_data))

Train size: 6811420
Val size: 756825


## 9. Context Window (Block Size)

The model does not see the entire dataset at once.

Instead, it learns from small chunks of text called "context windows".

Example:

Input:  "The cat sat" |
Target: "he cat sat "

The model learns to predict the next character given previous ones.


In [14]:
block_size = 128  # how many characters the model sees at once
batch_size = 32   # how many sequences per batch


def get_batch(split):
    data_split = train_data if split == 'train' else val_data
    
    ix = torch.randint(len(data_split) - block_size, (batch_size,))
    
    x = torch.stack([data_split[i:i+block_size] for i in ix])
    y = torch.stack([data_split[i+1:i+block_size+1] for i in ix])
    
    return x.to(device), y.to(device)


# Test it
xb, yb = get_batch('train')

print("Input shape:", xb.shape)
print("Target shape:", yb.shape)

Input shape: torch.Size([32, 128])
Target shape: torch.Size([32, 128])


## 10. Define Model Hyperparameters

We define key parameters controlling model size and training behaviour.


In [15]:
# Hyperparameters (GPU-friendly for 2080 Super)

batch_size = 32
block_size = 128

max_iters = 3000
eval_interval = 300
learning_rate = 3e-4

n_embd = 256
n_head = 4
n_layer = 4
dropout = 0.2

## 11. Define the Model

We build a simplified GPT architecture using embeddings, attention, and feedforward layers.


In [16]:
import torch.nn as nn
import torch.nn.functional as F

class Head(nn.Module):
    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        
        self.register_buffer("tril", torch.tril(torch.ones(block_size, block_size)))
        
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        
        k = self.key(x)
        q = self.query(x)
        
        wei = q @ k.transpose(-2, -1) * (C ** -0.5)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)
        
        v = self.value(x)
        out = wei @ v
        return out


class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.dropout(self.proj(out))
        return out


class FeedForward(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)


class Block(nn.Module):
    def __init__(self):
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedForward()
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x


class GPT(nn.Module):
    def __init__(self):
        super().__init__()
        
        self.token_embedding = nn.Embedding(vocab_size, n_embd)
        self.position_embedding = nn.Embedding(block_size, n_embd)
        
        self.blocks = nn.Sequential(*[Block() for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        
        tok_emb = self.token_embedding(idx)
        pos_emb = self.position_embedding(torch.arange(T, device=device))
        
        x = tok_emb + pos_emb
        x = self.blocks(x)
        x = self.ln_f(x)
        
        logits = self.lm_head(x)
        
        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)
        
        return logits, loss

    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:]
            logits, _ = self(idx_cond)
            
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        
        return idx

## 12. Initialize Model

We create the model and move it to GPU.


In [17]:
model = GPT().to(device)

print(sum(p.numel() for p in model.parameters()) / 1e6, "M parameters")

3.228236 M parameters


## 13. Training Setup

We define the optimizer used to train the model.


In [18]:
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

## 14. Estimate Loss

We periodically evaluate the model on training and validation data to monitor learning.


In [19]:
@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    
    for split in ['train', 'val']:
        losses = torch.zeros(200)
        
        for k in range(200):
            X, Y = get_batch(split)
            _, loss = model(X, Y)
            losses[k] = loss.item()
        
        out[split] = losses.mean()
    
    model.train()
    return out

## 15. Training Loop

We train the model by repeatedly sampling batches and updating weights.


In [20]:
for iter in range(max_iters):
    
    # Evaluate loss occasionally
    if iter % eval_interval == 0:
        losses = estimate_loss()
        print(f"Step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")
    
    # Get batch
    xb, yb = get_batch('train')
    
    # Forward pass
    logits, loss = model(xb, yb)
    
    # Backprop
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

Step 0: train loss 4.4819, val loss 4.4907
Step 300: train loss 2.2792, val loss 2.4172
Step 600: train loss 1.9449, val loss 2.1297
Step 900: train loss 1.7319, val loss 1.9530
Step 1200: train loss 1.6072, val loss 1.8607
Step 1500: train loss 1.5251, val loss 1.7890
Step 1800: train loss 1.4660, val loss 1.7294
Step 2100: train loss 1.4189, val loss 1.7010
Step 2400: train loss 1.3864, val loss 1.6697
Step 2700: train loss 1.3497, val loss 1.6532


## 16. Generate Text

We sample from the trained model to generate new text.


In [21]:
context = torch.zeros((1, 1), dtype=torch.long, device=device)

generated = model.generate(context, max_new_tokens=500)

print(decode(generated[0].tolist()))


 That shall not come make ofference offiming your delightens walks in things let up! 2:37 For Zels, had is measions, desinumury now that I amural apponsed founds, and the Lord, if be made sumpisie, of Siv Cienioner. 2 God pletwormoni Medipenis as in it be in evermanted evisiary, suon of my wises he of my into Petreserver and han omber, evenged the strengerlor abrased affore as prope35 And the wasten from man assegether goods of For themselves of Puccoraoar, or I will streems need a flespite, nor


## 17. Save Model

We store the trained model for future use.


In [22]:
torch.save(model.state_dict(), "gpt_model.pth")